# COLOCACIONES
Las **colocaciones** son combinaciones de palabras que ocurren juntas con más frecuencia de lo esperado por el azar. Ejemplos de colocaciones comunes son:
- "máquina de café"
- "nueva york"
- "clima cálido"

In [1]:
import nltk
nltk.download('book')
from nltk.book import *
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

[nltk_data] Downloading collection 'book'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package brown to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/brown.zip.
[nltk_data]    | Downloading package chat80 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/chat80.zip.
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package conll2000 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/conll2000.zip.
[nltk_data]    | Downloading package conll2002 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/conll2002.zip.
[nltk_data]    | Downloading package dependency_treebank to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping corpora/dependency_treebank.zip.
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


In [2]:
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
stop_words = set(stopwords.words('english'))
words = [word.lower() for word in text1 if word.isalpha()]
filtered_words = [word for word in words if word not in stop_words]
print(filtered_words[:10])

['moby', 'dick', 'herman', 'melville', 'etymology', 'supplied', 'late', 'consumptive', 'usher', 'grammar']


In [4]:
filtered_bigrams = list(bigrams(filtered_words))
filtered_bigrams[:10]

[('moby', 'dick'),
 ('dick', 'herman'),
 ('herman', 'melville'),
 ('melville', 'etymology'),
 ('etymology', 'supplied'),
 ('supplied', 'late'),
 ('late', 'consumptive'),
 ('consumptive', 'usher'),
 ('usher', 'grammar'),
 ('grammar', 'school')]

In [5]:
filtered_bigram_dist = FreqDist(filtered_bigrams)
filtered_bigram_dist

FreqDist({('sperm', 'whale'): 182, ('white', 'whale'): 106, ('moby', 'dick'): 84, ('old', 'man'): 81, ('captain', 'ahab'): 64, ('right', 'whale'): 57, ('mast', 'head'): 49, ('whale', 'ship'): 37, ('mast', 'heads'): 37, ('ye', 'see'): 37, ...})

In [6]:
threshold = 5
filtered_words = [word for word in filtered_words if len(word)>threshold]
filtered_word_dist = FreqDist(filtered_words)
filtered_word_dist

FreqDist({'though': 384, 'captain': 329, 'seemed': 283, 'whales': 268, 'queequeg': 252, 'little': 249, 'starbuck': 198, 'almost': 195, 'chapter': 173, 'pequod': 173, ...})

# CREAMOS UN DATAFRAME CON LOS VALORES DE FILTRED_BIGRAM_DIST

In [7]:
df = pd.DataFrame()
df['bi_gram'] = list(set(filtered_bigrams))
df['word_0'] = df['bi_gram'].apply(lambda x: x[0])
df['word_1'] = df['bi_gram'].apply(lambda x: x[1])
df['bi_gram_freq'] = df['bi_gram'].apply(lambda x: filtered_bigram_dist[x])
df['word_0_freq'] = df['word_0'].apply(lambda x: filtered_word_dist[x])
df['word_1_freq'] = df['word_1'].apply(lambda x: filtered_word_dist[x])
df

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq
0,"(one, step)",one,step,2,0,0
1,"(savor, given)",savor,given,1,0,0
2,"(two, plump)",two,plump,1,0,0
3,"(grow, fainter)",grow,fainter,1,0,2
4,"(desperate, scene)",desperate,scene,1,12,0
...,...,...,...,...,...,...
98017,"(first, class)",first,class,1,0,0
98018,"(cannon, bows)",cannon,bows,1,6,0
98019,"(sudden, bodily)",sudden,bodily,1,43,26
98020,"(say, examples)",say,examples,1,0,4


# APLICAMOS FORMULA PARA CALCULAR COLOCACIONES

# Pointwise Mutual Information (PMI)
Una métrica basada en _teoria de la información_ para encontrar **Collocations**.

$$
PMI = \log\left(\frac{P(w_1, w_2)}{P(w_1)P(w_2)}\right)
$$

In [8]:
# Suponiendo que N es la suma total de las frecuencias de bigramas
N = df['bi_gram_freq'].sum()

In [9]:
# Para evitar división por cero y log de cero, usa np.maximum con un valor mínimo (por ejemplo, 1)
df['PMI'] = df.apply(lambda x: np.log2(
    np.maximum(x['bi_gram_freq'], 1) * N /
    (np.maximum(x['word_0_freq'], 1) * np.maximum(x['word_1_freq'], 1))
), axis=1)

In [10]:
# Logaritmo solo del bi_gram_freq (esto es solo log transform normal)
df['log(bi_gram_freq)'] = df['bi_gram_freq'].apply(lambda x: np.log2(np.maximum(x, 1)))
df

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI,log(bi_gram_freq)
0,"(one, step)",one,step,2,0,0,17.753138,1.0
1,"(savor, given)",savor,given,1,0,0,16.753138,0.0
2,"(two, plump)",two,plump,1,0,0,16.753138,0.0
3,"(grow, fainter)",grow,fainter,1,0,2,15.753138,0.0
4,"(desperate, scene)",desperate,scene,1,12,0,13.168176,0.0
...,...,...,...,...,...,...,...,...
98017,"(first, class)",first,class,1,0,0,16.753138,0.0
98018,"(cannon, bows)",cannon,bows,1,6,0,14.168176,0.0
98019,"(sudden, bodily)",sudden,bodily,1,43,26,6.626434,0.0
98020,"(say, examples)",say,examples,1,0,4,14.753138,0.0


In [11]:
df.sort_values(by = 'PMI', ascending=False)

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI,log(bi_gram_freq)
53297,"(sperm, whale)",sperm,whale,182,0,0,24.260933,7.507795
55729,"(white, whale)",white,whale,106,0,0,23.481059,6.727920
69228,"(moby, dick)",moby,dick,84,0,0,23.145456,6.392317
2022,"(old, man)",old,man,81,0,0,23.092988,6.339850
32195,"(right, whale)",right,whale,57,0,0,22.586028,5.832890
...,...,...,...,...,...,...,...,...
87555,"(whales, seemed)",whales,seemed,1,268,283,0.542391,0.000000
28718,"(captain, little)",captain,little,1,329,249,0.431193,0.000000
94418,"(little, captain)",little,captain,1,249,329,0.431193,0.000000
3793,"(though, queequeg)",though,queequeg,1,384,252,0.190896,0.000000


# CALCULO DE PMI PARA COLOCACIONES USANDO NTLK

In [12]:
stop_words = set(stopwords.words('english'))
words = [word.lower() for word in text1 if word.isalpha()]
filtered_words = [word for word in words if word not in stop_words]
print(filtered_words[:10])

['moby', 'dick', 'herman', 'melville', 'etymology', 'supplied', 'late', 'consumptive', 'usher', 'grammar']


In [13]:
from  nltk.collocations import *
bigram_measures = nltk.collocations.BigramAssocMeasures()
finder = BigramCollocationFinder.from_words(filtered_words)
finder.apply_freq_filter(20)

In [14]:
finder.nbest(bigram_measures.pmi,10)

[('moby', 'dick'),
 ('quarter', 'deck'),
 ('mast', 'heads'),
 ('mr', 'starbuck'),
 ('thou', 'art'),
 ('aye', 'aye'),
 ('never', 'mind'),
 ('captain', 'peleg'),
 ('mast', 'head'),
 ('let', 'us')]